[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [5]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [9]:
import torch
import torch.nn as nn
import math

In [26]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):

      super().__init__()

      assert d_model % num_heads == 0 # ensure even division of dimensions across heads

      # store dimensions
      self.d_model = d_model
      self.num_heads = num_heads
      self.d_k = d_model // num_heads # dimensions per head

      # init linear layers W_q, W_k, W_v, W_o (pass in input / output sizes)
      self.W_q = nn.Linear(d_model, d_model)
      self.W_k = nn.Linear(d_model, d_model)
      self.W_v = nn.Linear(d_model, d_model)
      self.W_o = nn.Linear(d_model, d_model)


    def forward(self, Q, K, V):

      # apply linear projection to input tensor
      Q = self.W_q(Q)
      K = self.W_k(K)
      V = self.W_v(V)

      # reshape
      batch_size = Q.shape[0]

      seq_q = Q.shape[1]
      seq_k = K.shape[1]
      seq_v = V.shape[1]

      Q = Q.view(batch_size, seq_q, self.num_heads, self.d_k) # reshaping using .view
      K = K.view(batch_size, seq_k, self.num_heads, self.d_k)
      V = V.view(batch_size, seq_v, self.num_heads, self.d_k)

      Q = Q.transpose(1, 2) # easier for attention if num_heads comes before seq_len
      K = K.transpose(1, 2)
      V = V.transpose(1, 2)

      # compute attention scores
      scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

      # softmax
      attn_weights = torch.softmax(scores, dim=-1)

      # weighted combination of V
      output = torch.matmul(attn_weights, V)

      # concatenate heads
      output = output.transpose(1, 2) # transpose back
      output = output.contiguous().view( # flatten heads + d_k together
          batch_size,
          seq_q,
          self.d_model
      )

      output = self.W_o(output) # final output projection
      return output


In [27]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [28]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (8.0ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (0.9ms)
  ✅ [3/6] Numerical correctness vs reference (3.1ms)
  ✅ [4/6] Gradient flow (3.4ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (1.3ms)
  ✅ [6/6] Different heads give different outputs (1.9ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (18.6ms total)
  Progress saved. Run status() to see your dashboard.

